## 3.1 Validation Sample Entertainment Videos

**Author:** Kristina Aleksandrovna Pedersen

The code shows how the manual validation samples for entertainment videos we're generated and distributed across RAs.

In [5]:
#Loading relevant modules
import pandas as pd
import re
import os
import shutil
from glob import glob
#from functions import *
import random

In [6]:
#load the (cleaned/parsed) entertainment metadata
df = pd.read_pickle('entertainment/ent_video_metadata_clean.pkl')
print(df.shape)
df.head()

(750, 21)


,video_id,video_desc,video_timestamp,video_dur_secs,author_id,user_name,author_verified,likes,shares,comments,...,saved,location,cat_code,labels,url,sharelink,version,vpn_location,video_datetime,video_date
0,7448072578289143086,hi,1734139536,9.0,6915139520857949190,stevenisanonymous,False,519600.0,24100.0,3010.0,...,36583,US,110.0,Lip-sync_Lip-sync_Performance,https://www.tiktok.com/@stevenisanonymous/vide...,https://www.tiktok.com/share/video/74480725782...,batch_1.pkl,Berlin,2024-12-14 01:25:36+00:00,2024-12-14
1,7447412950228569367,,1733985962,15.0,7396454683768472608,anis....._,False,1300000.0,450500.0,20100.0,...,148086,DZ,119.0,Finger Dance & Basic Dance_Singing & Dancing_T...,https://www.tiktok.com/@anis....._/video/74474...,https://www.tiktok.com/share/video/74474129502...,batch_1.pkl,Berlin,2024-12-12 06:46:02+00:00,2024-12-12
2,7452178491488898305,Who should i pronk next?,1735095555,17.0,7154829325820593178,jepitot_,False,7800000.0,371000.0,30700.0,...,578393,PH,104.0,Pranks_Comedy_Performance,https://www.tiktok.com/@jepitot_/video/7452178...,https://www.tiktok.com/share/video/74521784914...,batch_1.pkl,Berlin,2024-12-25 02:59:15+00:00,2024-12-25
3,7443268490208841016,After ?💀#velocityedit #makeup,1733021001,11.0,7168213111505175558,itsinctina,False,185500.0,4618.0,533.0,...,20952,CA,105.0,Diary & VLOG_Daily Life_Lifestyle,https://www.tiktok.com/@itsinctina/video/74432...,https://www.tiktok.com/share/video/74432684902...,batch_1.pkl,Berlin,2024-12-01 02:43:21+00:00,2024-12-01
4,7448718017300598021,каждый день новые видео ✅ инст lazerniy_orel,1734289820,19.0,6799921696088081413,jumm_video,False,121500.0,22700.0,190.0,...,6544,MD,104.0,Comedy_Performance,https://www.tiktok.com/@jumm_video/video/74487...,https://www.tiktok.com/share/video/74487180173...,batch_1.pkl,Berlin,2024-12-15 19:10:20+00:00,2024-12-15


#### Create validation sample files

In [ ]:
num_ras = 6 #set number of coders
sample_size = 50 #set sample size for each individual

In [31]:
#empty list to hold data
dfs = []
#Loop through each coder
for ra in range(1, num_ras+1):
    #draw random sample of videos into a dataframe
    sample = df.sample(sample_size, random_state = 123)
    #assign the coder to the videos
    sample['ra_num'] = ra

    #drop the in-sample videos from general data file
    df.drop(index = sample.index, inplace = True)
    #append sample dataframe
    dfs.append(sample)

In [35]:
#compile into one dataframe
sampledf = pd.concat(dfs, ignore_index = True)
#keep selected columns (Coder ID, user handle, user ID, video ID, posting date, video URL)
sampledf = sampledf[['ra_num', 'user_name', 'author_id', 'video_id' , 'video_date', 'url']]
#empty column for coder to indicate whether video is political
#Note: sensitive content column was created later, as we identified such videos in the validation process
sampledf['political'] = None
print(sampledf.shape)
sampledf.head(3)

(300, 7)


,ra_num,user_name,author_id,video_id,video_date,url,political
0,1,ketikajazan,7080486714355024898,7445303814598479120,2024-12-06,https://www.tiktok.com/@ketikajazan/video/7445...,None
1,1,.multi.fandom123,7317272065152451616,7447224671474388226,2024-12-11,https://www.tiktok.com/@.multi.fandom123/video...,None
2,1,jean__kadyma099,7388632711642874913,7451009742132006166,2024-12-21,https://www.tiktok.com/@jean__kadyma099/video/...,None


In [36]:
#Save each sample in seperate file, ready for manual validation by human coder
for ra in sampledf["ra_num"].unique():
    subdf = sampledf.query(f"ra_num == {ra}")
    subdf.to_excel(f"samples/ra_{ra}_entertainment.xlsx", index = False)

#### Assign samples to human coders

In [ ]:
#Dictionary of Coder IDs (emails that are anonymized here)
coders = dict.fromkeys(['anonymized coder ID 1', 'anonymized coder ID 2', 'anonymized coder ID 3',
                       'anonymized coder ID 4', 'anonymized coder ID 5', 'anonymized coder ID 6'], 
                       None)

In [29]:
#Shuffle coder number used in samples
ra_num = [1,2,3,4,5,6]
random.shuffle(ra_num)

#for each human coder ID randomly assign the person a coder number adjacent to a sample file 
for ra in coders:
    coders[ra] =  random.choice(ra_num)
    ra_num.pop()